# Agent Team Starter v0.9.1 — Colab launcher

This notebook installs the selected ZIP, checks it, loads optional Colab Secrets without printing secret values, runs offline tests, and lets you submit a task. **No live model call is made until you run the final cell.** Provider usage may consume quota or be billable depending on account settings.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, os

uploaded = files.upload()  # Select agent_team_starter_v0.9.1-rc15.zip
zip_names = [name for name in uploaded if name.endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("Upload exactly one .zip archive.")
archive = Path(zip_names[0])
root = Path("/content/agent_team_starter_v0.9").resolve()
with zipfile.ZipFile(archive) as zf:
    for info in zf.infolist():
        target = (root.parent / info.filename).resolve()
        if target != root.parent and root not in target.parents and target != root:
            raise ValueError("Unsafe archive path detected.")
    if zf.testzip() is not None:
        raise ValueError("Archive integrity check failed.")
    zf.extractall(root.parent)
project = root
if not (project / "requirements.txt").is_file() or not (project / "agent_team").is_dir():
    raise FileNotFoundError("Expected v0.9 project layout was not found.")
print("Archive verified and extracted.")

In [ ]:
%cd /content/agent_team_starter_v0.9
%pip -q install -r requirements.txt

In [ ]:
import os
try:
    from google.colab import userdata
    for name in ("GEMINI_API_KEY", "OPENROUTER_API_KEY", "GEMINI_MODEL", "OPENROUTER_FREE_MODELS"):
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
except Exception:
    pass
print("Configured provider names:", [n for n in ("GEMINI_API_KEY", "OPENROUTER_API_KEY") if os.getenv(n)])
print("Secret values are never printed.")

In [ ]:
!python run_tests.py

In [ ]:
!python -m agent_team.cli --check-config

## Run a task
Run this final cell only after checking the provider’s current quota and billing terms. This makes live API requests if a key is configured. No API key is required for offline tests.

In [ ]:
from agent_team.team import run_team
from agent_team.router import FreeFirstRouter

task = input("Task for the agent team: ").strip()
if not task:
    raise ValueError("Task cannot be empty.")
result = run_team(task, FreeFirstRouter())
print("FINAL DRAFT\n")
print(result["final"])
print("\nREVIEW\n")
print(result["review"]["text"])
print("\nTRACE\n")
print(result["trace"])

## Benchmark scorer (local, no API call)
Upload a completed benchmark CSV with the required columns, then score it. A declared security failure forces rejection.


In [ ]:
!python evaluations/benchmark_score.py benchmark_results.csv


## Security limitation
URL validation rejects private DNS answers and local targets, but DNS rebinding is not fully prevented because validation and the HTTP connection resolve separately. Do not expose the fetcher as a public service without network egress controls.


## v0.9.1-rc5 proxy configuration hardening and evaluation
The package now includes 12 deterministic reference cases, financial-style half-up rounding, rejection of NaN/infinity, and tolerance validation. Offline checks do not establish live model quality.


This candidate adds a 12-task realistic benchmark specification. Live model quality, latency and cost have not been measured.


The trusted egress proxy URL is validated for syntax, embedded credentials, ports, and unexpected URL components. This does not verify the proxy's destination-IP policy. Live proxy/security tests remain a separate deployment gate.

The fetcher uses an explicit requests session with trust_env=False so ambient NO_PROXY and proxy variables cannot silently override the configured egress proxy. This is tested with mocked requests; real network egress policy remains unverified.

## v0.9.1-rc7 — Anthropic/Google skill patterns and offline validation
The project adds a standard-library-only skill metadata check and three portable skills: prompt-injection defense, fair framework evaluation, and Google agent-stack assessment. The tool radar now covers official Anthropic Agent Skills/Claude Code and Google Gemini CLI/ADK/OSS-Fuzz/OSV-Scanner repositories. Hosted model access, free quotas, privacy, and project license are separate questions. No external packages are added by these changes.


## Validate local skills (offline, no API)
Run this after the test suite to check required metadata and folder-name consistency. It is not a complete specification validator or a safety certification.


In [ ]:
!python -m agent_team.skill_lint .
!python -m agent_team.skill_lint . --json